# 36-bus graph comparison: IL vs IL + RL

Does the graph representation of the observation matter on IEEE-36 (`l2rpn_wcci_2020`), and does the
difference persist after RL fine-tuning? The 12 representations of the graph comparison, all with the
Complete node features: 11 graphs with a GCN and per-substation readout (`model=gcn_substation_readout`),
among them `no_edges` (the nodes without any edges, structure control), and `flat` (RLlib's default MLP on
the flattened features).

Per representation and training seed (0–4):

- **Junior (IL)**: the best HPO trial of `graph36_<obs>`, retrained with the seed.
- **Senior (IL + RL)**: PPO (`senior_warmup`, 200k agent steps) from that Junior's weights. Each Senior
  ran on CPU and GPU in parallel; the first to finish was kept (`eval/senior_s<seed>/trained_on.txt`).
  RL setup since `*_v2`: every rollout worker plays the train chronics in its own random order, every
  evaluation plays all 144 val chronics, and the exported checkpoint is the one with the highest val
  survival (`eval/senior_s<seed>/checkpoint_selection.json`). Senior val survival is therefore slightly
  optimistic; **test is the unbiased split**. `2026_11_10_IEEE36_graph_comparison_senior` (4 representations)
  used sorted chronics, rotating 18-chronic evaluations and the last checkpoint.

Both are rolled out greedily (argmax) on `l2rpn_wcci_2020_val` and `l2rpn_wcci_2020_test` (144 chronics
each, the same chronics and env seeds for every agent). Run: `slurm/unicluster/graph_comparison_36bus_senior.sh`.

**Metrics.** Per seed: *survived steps* = mean over chronics of steps survived / chronic length;
*survived episodes* = share of chronics survived to the end. Tables: mean ± SD over seeds (sample SD).
Only seeds with all 144 chronics of a split count; missing or incomplete seeds are listed below.

**Data.** `EXPERIMENT` selects `results/senior/<EXPERIMENT>/`:
```
<obs>/junior_s<seed>/rollout/episodes_{valenv,testenv}_*.csv       Junior rollouts (one file per rollout task/process)
<obs>/eval/senior_s<seed>/rollout/episodes_{valenv,testenv}_*.csv  Senior rollouts
```
A local copy needs one rsync (only the rollout tables, a few MB):
```
rsync -avm --include '*/' --include 'episodes_*.csv' --include 'trained_on.txt' --include 'checkpoint_selection.json' --exclude '*' \
    <cluster>:<repo>/results/senior/<EXPERIMENT>/ results/senior/<EXPERIMENT>/
```

In [5]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))

from analysis.episodes import survival_over_seeds, survival_per_seed

# ── Experiment ──
EXPERIMENT = "2026_11_11_IEEE36_graph_comparison_senior_v2"  # results/senior/<EXPERIMENT>/
BASELINE_EXPERIMENT = "2026_11_10_IEEE36_graph_comparison_senior"  # earlier RL setup, same Juniors; None to skip
OBS_SPACES = ["no_edges", "substation", "complete", "elements", "heterogeneous", "lodf", "ptdf", "zbus",
              "elements_lodf", "substation_ptdf", "substation_zbus", "flat"]
SEEDS = range(5)
SPLITS = {"val": "valenv", "test": "testenv"}  # Grid2Op env -> tag of the episode files
N_CHRONICS = 144  # chronics per env split
STAGES = {"IL": "junior_s{seed}", "IL + RL": "eval/senior_s{seed}"}  # stage (Junior, Senior) -> run folder

EXP_DIR = ROOT / "results/senior" / EXPERIMENT
assert EXP_DIR.exists(), f"{EXP_DIR} not found: copy it from the cluster (rsync command above)"

## Rollouts per seed

Loads every seed's episode tables (the rollout tasks of a split appended) and keeps the complete seeds.
Missing or incomplete seeds are listed and left out of all tables below.

In [6]:
def rollout_folders(obs: str, stage: str) -> dict[int, Path]:
    """seed -> rollout folder of one representation and stage."""
    return {seed: EXP_DIR / obs / STAGES[stage].format(seed=seed) / "rollout" for seed in SEEDS}


def incomplete_seeds(per_seed: pd.DataFrame) -> dict[int, int]:
    """seed -> chronics found, for every seed of SEEDS without all N_CHRONICS chronics."""
    found = per_seed["chronics"].reindex(SEEDS, fill_value=0)
    return {seed: int(n) for seed, n in found.items() if n != N_CHRONICS}


PER_SEED = {}  # (stage, split, obs) -> one row per complete seed (index seed, SurvivalMetrics columns)
missing = []
for stage in STAGES:
    for split, tag in SPLITS.items():
        for obs in OBS_SPACES:
            per_seed = survival_per_seed(rollout_folders(obs, stage), tag)
            gaps = incomplete_seeds(per_seed)
            missing += [f"{stage}, {split}, {obs} s{seed}: {n}/{N_CHRONICS} chronics" for seed, n in gaps.items()]
            PER_SEED[(stage, split, obs)] = per_seed.drop(index=[s for s in gaps if s in per_seed.index])

print("Missing or incomplete, left out:" if missing else "All seeds complete.")
for line in missing:
    print("  " + line)

Missing or incomplete, left out:
  IL + RL, val, no_edges s4: 72/144 chronics


## Survival per env split

One table per split. Rows: representations; columns: IL (Junior) and IL + RL (Senior).
Mean ± SD over the complete seeds, in %.

In [7]:
def pm(mean: float, sd: float) -> str:
    """'mean ± SD' with one decimal; SD left out for one seed (NaN), '–' without seeds."""
    if np.isnan(mean):
        return "–"
    return f"{mean:.1f}" if np.isnan(sd) else f"{mean:.1f} ± {sd:.1f}"


def stage_columns(per_seed: pd.DataFrame, stage: str) -> dict[tuple[str, str], str | int]:
    """(stage, column) -> value: step and episode survival (mean ± SD over seeds) and the seed count.

    Args:
        per_seed: One row per seed with `mean_survival_pct` and `full_survival_pct`.
        stage: Key of STAGES, the top level of the column names.
    """
    s = survival_over_seeds(per_seed)
    return {(stage, "Survived steps (%)"): pm(s.steps_pct, s.steps_sd),
            (stage, "Survived episodes (%)"): pm(s.episodes_pct, s.episodes_sd),
            (stage, "Seeds"): s.seeds}


def split_table(split: str) -> pd.DataFrame:
    """One row per representation, IL and IL + RL survival side by side (two-level columns: stage, metric)."""
    rows = {obs: {k: v for stage in STAGES for k, v in stage_columns(PER_SEED[(stage, split, obs)], stage).items()}
            for obs in OBS_SPACES}
    table = pd.DataFrame.from_dict(rows, orient="index")
    table.columns = pd.MultiIndex.from_tuples(table.columns)
    return table.rename_axis("Representation")


def show(table: pd.DataFrame, title: str) -> None:
    """Display a table and print it as markdown (columns flattened to 'stage: metric') for copying."""
    print(title)
    display(table)
    flat = table.set_axis([f"{stage}: {metric}" for stage, metric in table.columns], axis=1)
    print(flat.to_markdown(), "\n")


for split in SPLITS:
    show(split_table(split), f"{split} env")

val env


IL                              \
                Survived steps (%) Survived episodes (%) Seeds   
Representation                                                   
no_edges                32.0 ± 6.0            14.3 ± 4.7     5   
ptdf                    38.8 ± 3.3            20.7 ± 4.9     5   
substation              34.7 ± 3.5            16.4 ± 3.0     5   
substation_ptdf         44.2 ± 5.2            24.9 ± 5.7     5   

                           IL + RL                              
                Survived steps (%) Survived episodes (%) Seeds  
Representation                                                  
no_edges                58.5 ± 2.8            38.4 ± 2.8     4  
ptdf                    42.4 ± 8.6            23.5 ± 8.1     5  
substation              58.8 ± 9.5            36.8 ± 8.5     5  
substation_ptdf         59.6 ± 8.9            39.9 ± 9.5     5

| Representation   | IL: Survived steps (%)   | IL: Survived episodes (%)   |   IL: Seeds | IL + RL: Survived steps (%)   | IL + RL: Survived episodes (%)   |   IL + RL: Seeds |
|:-----------------|:-------------------------|:----------------------------|------------:|:------------------------------|:---------------------------------|-----------------:|
| no_edges         | 32.0 ± 6.0               | 14.3 ± 4.7                  |           5 | 58.5 ± 2.8                    | 38.4 ± 2.8                       |                4 |
| ptdf             | 38.8 ± 3.3               | 20.7 ± 4.9                  |           5 | 42.4 ± 8.6                    | 23.5 ± 8.1                       |                5 |
| substation       | 34.7 ± 3.5               | 16.4 ± 3.0                  |           5 | 58.8 ± 9.5                    | 36.8 ± 8.5                       |                5 |
| substation_ptdf  | 44.2 ± 5.2               | 24.9 ± 5.7                  |           5 | 59.6 ± 8.9        

IL                              \
                Survived steps (%) Survived episodes (%) Seeds   
Representation                                                   
no_edges                32.5 ± 6.7            12.8 ± 5.7     5   
ptdf                    39.5 ± 1.4            17.8 ± 2.1     5   
substation              33.7 ± 5.2            13.6 ± 4.5     5   
substation_ptdf         42.6 ± 4.6            21.5 ± 4.5     5   

                           IL + RL                              
                Survived steps (%) Survived episodes (%) Seeds  
Representation                                                  
no_edges                60.7 ± 4.3            35.0 ± 6.1     5  
ptdf                    45.8 ± 6.7            22.9 ± 7.1     5  
substation              61.0 ± 7.9            36.2 ± 5.7     5  
substation_ptdf         61.4 ± 8.8           37.8 ± 10.1     5

| Representation   | IL: Survived steps (%)   | IL: Survived episodes (%)   |   IL: Seeds | IL + RL: Survived steps (%)   | IL + RL: Survived episodes (%)   |   IL + RL: Seeds |
|:-----------------|:-------------------------|:----------------------------|------------:|:------------------------------|:---------------------------------|-----------------:|
| no_edges         | 32.5 ± 6.7               | 12.8 ± 5.7                  |           5 | 60.7 ± 4.3                    | 35.0 ± 6.1                       |                5 |
| ptdf             | 39.5 ± 1.4               | 17.8 ± 2.1                  |           5 | 45.8 ± 6.7                    | 22.9 ± 7.1                       |                5 |
| substation       | 33.7 ± 5.2               | 13.6 ± 4.5                  |           5 | 61.0 ± 7.9                    | 36.2 ± 5.7                       |                5 |
| substation_ptdf  | 42.6 ± 4.6               | 21.5 ± 4.5                  |           5 | 61.4 ± 8.8        

## New vs earlier RL setup

Senior of this experiment minus the Senior of `BASELINE_EXPERIMENT` for the same seed. Both start from the
same Junior (copied with `JUNIOR_EXPERIMENT`), so the difference is the RL setup alone (shuffled training
chronics, full evaluation, best-val checkpoint). In percentage points, mean ± SD over the seeds complete in
both experiments; only representations of both experiments.

In [ ]:
SURVIVAL_COLUMNS = ["mean_survival_pct", "full_survival_pct"]


def paired_difference(before: pd.DataFrame, after: pd.DataFrame) -> pd.DataFrame:
    """Per-seed `after - before` of step and episode survival, for the seeds in both tables (index seed)."""
    seeds = before.index.intersection(after.index)
    return after.loc[seeds, SURVIVAL_COLUMNS] - before.loc[seeds, SURVIVAL_COLUMNS]


def senior_per_seed(exp_dir: Path, split: str, obs: str) -> pd.DataFrame:
    """Complete Senior seeds of one experiment, split and representation (index seed)."""
    folders = {seed: exp_dir / obs / STAGES["IL + RL"].format(seed=seed) / "rollout" for seed in SEEDS}
    per_seed = survival_per_seed(folders, SPLITS[split])
    return per_seed[per_seed["chronics"] == N_CHRONICS]


if BASELINE_EXPERIMENT:
    BASELINE_DIR = ROOT / "results/senior" / BASELINE_EXPERIMENT
    assert BASELINE_DIR.exists(), f"{BASELINE_DIR} not found: copy it from the cluster (rsync command above)"
    common = [obs for obs in OBS_SPACES if (BASELINE_DIR / obs).exists()]
    for split in SPLITS:
        rows = {}
        for obs in common:
            diff = paired_difference(senior_per_seed(BASELINE_DIR, split, obs), PER_SEED[("IL + RL", split, obs)])
            s = survival_over_seeds(diff)
            rows[obs] = {"Δ survived steps (pp)": pm(s.steps_pct, s.steps_sd),
                         "Δ survived episodes (pp)": pm(s.episodes_pct, s.episodes_sd), "Pairs": s.seeds}
        table = pd.DataFrame.from_dict(rows, orient="index").rename_axis("Representation")
        print(f"Senior {EXPERIMENT} − {BASELINE_EXPERIMENT}, {split} env")
        display(table)
        print(table.to_markdown(), "\n")